# 05 · BioTrove-CLIP + per-pixel head (semantic segmentation)

Trains a per-pixel model: a pretrained ViT backbone (**BioTrove-CLIP ViT-B/16** by default, trained on ~40M plant/animal photos, MIT) + a light DPT-style decoder.
Every pixel gets one class: background / leaf / leaf_spot / powdery_mildew. Coverage % is then pixel counting.

- Model code: `semseg.py` (shared with `03_evaluate.ipynb` and `06_backbone_study.ipynb`).
- Data: the COCO export from `01_download_dataset.ipynb`. Instance polygons are turned into one class map per image (diseases painted over leaf).
- Runs go to `runs/semseg-<backbone>[-frozen]_data-v<N>/`. **Resuming is automatic**, as in `02_train.ipynb`.
- Other backbones for the comparison study (run later): set `BACKBONE` to `"clip-b16"`, `"dinov2-b14"` or `"imagenet-b16"`. Everything else stays identical.

In [ ]:
import json
import math
import os
import random
import time
from pathlib import Path

import numpy as np
import torch
from dotenv import load_dotenv
from torch.utils.data import DataLoader

import semseg

ROOT = Path.cwd()
load_dotenv(ROOT / ".env")
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

DATA_VERSION = f"v{os.environ['ROBOFLOW_VERSION']}"
DATA_NAME = f"{os.environ['ROBOFLOW_PROJECT']}-{DATA_VERSION}"  # e.g. leaf-dis-seg-v1
DATASET_DIR = ROOT / "datasets" / f"{DATA_NAME}-coco"
DEVICE = "cuda"

print("data:", DATA_NAME)
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## Settings

- `BACKBONE`: `"biotrove-b16"` (default), or for the comparison study `"clip-b16"` (the OpenAI CLIP model BioTrove started from), `"dinov2-b14"`, `"imagenet-b16"`.
- `FREEZE = True` trains only the decoder (tests how good the pretrained features are as they are); `False` fine-tunes the whole model with layer-wise lr decay.
- `BATCH = 4` at 1024 px uses roughly 20–25 GB with gradient checkpointing; lower it if the GPU is shared.
- `TRAIN_FRACTION < 1` trains on a random subset (quick tests only). `RUN_TAG` (e.g. `"_s1"` for another seed) starts a separate run.
- Each run trains for exactly `EPOCHS` (no early stopping), so different backbones are compared on equal terms; the best epoch by validation leaf/disease mIoU is kept.

In [ ]:
BACKBONE = "biotrove-b16"
FREEZE = False
IMGSZ = 1024
EPOCHS = 40
BATCH = 4
BACKBONE_LR = 1e-5  # last ViT block; earlier blocks get LAYER_DECAY× less per block
DECODER_LR = 1e-4
LAYER_DECAY = 0.75
WEIGHT_DECAY = 0.05
WARMUP_EPOCHS = 1
TRAIN_FRACTION = 1.0
SEED = 0
RUN_TAG = ""

SETTINGS = {k: v for k, v in dict(globals()).items() if k.isupper() and k not in {"ROOT", "DATASET_DIR", "DEVICE"} and isinstance(v, (str, int, float, bool))}
RUN_NAME = f"semseg-{BACKBONE}{'-frozen' if FREEZE else ''}_data-{DATA_VERSION}{RUN_TAG}"
RUN_DIR = ROOT / "runs" / RUN_NAME
LAST_CHECKPOINT = RUN_DIR / "checkpoint_last.pth"

# Decide what to do from the run folder: "new", "resume" or "finished"
state = json.loads((RUN_DIR / "state.json").read_text()) if (RUN_DIR / "state.json").exists() else None
if not LAST_CHECKPOINT.exists():
    ACTION = "new"
elif state and state["finished"]:
    ACTION = "finished"
else:
    ACTION = "resume"
if state and ACTION == "resume":
    print(f"last saved epoch: {state['epoch']} of {state['epochs']}")
print(BACKBONE, ACTION, "→", RUN_DIR)

## Labels

Class maps are written once to `datasets/<data>-coco/<split>/_semseg/*.png` (pixel values 0–3) and reused afterwards.

In [ ]:
items = {split: semseg.ensure_label_cache(DATASET_DIR / split) for split in ("train", "valid", "test")}
for split, it in items.items():
    print(f"{split:>5}: {len(it)} images")

import matplotlib.pyplot as plt
from PIL import Image

PALETTE = np.array([(0, 0, 0), (0, 200, 0), (230, 120, 0), (180, 0, 220)], np.uint8)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (img_path, label_path) in zip(axes, items["valid"][:3]):
    img = np.array(Image.open(img_path).convert("RGB"), float)
    cmap = np.array(Image.open(label_path))
    shown = np.where(cmap[..., None] > 0, 0.5 * img + 0.5 * PALETTE[cmap], img)
    ax.imshow(shown.astype(np.uint8)); ax.axis("off")
    ax.set_title(" ".join(f"{c}:{(cmap == i).mean() * 100:.1f}%" for i, c in enumerate(semseg.CLASSES) if i))

## Model

New runs start from the pretrained backbone (BioTrove weights are downloaded once from Hugging Face). Resumed runs load everything from `checkpoint_last.pth`.

In [ ]:
torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)

model = semseg.SemSegModel(BACKBONE, pretrained=ACTION == "new").to(DEVICE)
if FREEZE:
    model.backbone.requires_grad_(False)
else:
    model.backbone.set_grad_checkpointing(True)  # recompute activations in backward: ~half the memory

n_all = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{BACKBONE}: {n_all / 1e6:.1f}M parameters, {n_train / 1e6:.1f}M trainable")

## Train

Cross-entropy (weighted toward the rare spot/mildew pixels) + Dice loss, AdamW, warmup + cosine schedule, bf16 mixed precision.
After every epoch: validation IoU per class and image-level coverage MAE; the checkpoint is saved, so an interrupted run resumes from the last finished epoch (just run the notebook again).
`num_workers=0` because Windows data loaders often hang inside Jupyter.

In [ ]:
train_items = items["train"]
if TRAIN_FRACTION < 1:
    train_items = random.Random(SEED).sample(train_items, max(BATCH, int(len(train_items) * TRAIN_FRACTION)))

train_loader = DataLoader(semseg.SemSegDataset(train_items, IMGSZ, train=True), batch_size=BATCH, shuffle=True,
                          drop_last=True, num_workers=0, pin_memory=True, generator=torch.Generator().manual_seed(SEED))
val_loader = DataLoader(semseg.SemSegDataset(items["valid"], IMGSZ), batch_size=BATCH, num_workers=0, pin_memory=True)

class_weights = semseg.class_pixel_weights(items["train"]).to(DEVICE)
print("class weights:", dict(zip(semseg.CLASSES, class_weights.cpu().numpy().round(2))))

optimizer = torch.optim.AdamW(semseg.param_groups(model, BACKBONE_LR, DECODER_LR, LAYER_DECAY, WEIGHT_DECAY))
steps_per_epoch = len(train_loader)
total_steps, warmup_steps = EPOCHS * steps_per_epoch, WARMUP_EPOCHS * steps_per_epoch
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda s: (s + 1) / warmup_steps if s < warmup_steps
                                              else 0.5 * (1 + math.cos(math.pi * (s - warmup_steps) / max(1, total_steps - warmup_steps))))

start_epoch, best, history = 0, -1.0, []
if ACTION == "resume":
    ck = torch.load(LAST_CHECKPOINT, map_location="cpu", weights_only=False)
    model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optimizer"]); scheduler.load_state_dict(ck["scheduler"])
    start_epoch, best, history = ck["epoch"], ck["best_miou_fg"], ck["history"]
    del ck
RUN_DIR.mkdir(parents=True, exist_ok=True)

if ACTION == "finished":
    print("This run already finished; nothing to train. Use RUN_TAG for a new run.")
else:
    for epoch in range(start_epoch, EPOCHS):
        model.train()
        t0, running = time.time(), 0.0
        for step, (x, y) in enumerate(train_loader):
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.bfloat16):
                logits = model(x)
            loss = semseg.seg_loss(logits.float(), y, class_weights)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            running += loss.item()
            if step % 100 == 0:
                print(f"  epoch {epoch + 1} step {step}/{steps_per_epoch} loss {loss.item():.3f}", flush=True)

        val = semseg.evaluate(model, val_loader, DEVICE)
        row = {"epoch": epoch + 1, "train_loss": running / steps_per_epoch, "val_miou": val["miou"], "val_miou_fg": val["miou_fg"],
               **{f"val_iou_{c}": v for c, v in val["iou"].items()},
               **{f"val_coverage_mae_{d}": v for d, v in val["coverage_mae"].items()},
               "minutes": (time.time() - t0) / 60}
        history.append(row)
        is_best = val["miou_fg"] > best
        best = max(best, val["miou_fg"])
        semseg.save_checkpoint(RUN_DIR, model, optimizer, scheduler, epoch + 1, EPOCHS, best, history, SETTINGS, is_best)
        print(f"epoch {epoch + 1}/{EPOCHS}  loss {row['train_loss']:.3f}  val mIoU(leaf+disease) {val['miou_fg']:.3f}  "
              + "  ".join(f"{c} {v:.3f}" for c, v in val["iou"].items() if c != "background")
              + f"  coverage MAE spot {val['coverage_mae']['leaf_spot']:.2f} / mildew {val['coverage_mae']['powdery_mildew']:.2f} pp"
              + ("  ← best" if is_best else "") + f"  ({row['minutes']:.1f} min)", flush=True)

## Results

`checkpoint_best.pth` (best validation leaf/disease mIoU) is what `03_evaluate.ipynb` and `06_backbone_study.ipynb` load.

In [ ]:
import csv

import matplotlib.pyplot as plt

rows = list(csv.DictReader(open(RUN_DIR / "history.csv")))
ep = [int(r["epoch"]) for r in rows]
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
ax[0].plot(ep, [float(r["train_loss"]) for r in rows]); ax[0].set_title("train loss")
for c in semseg.CLASSES[1:]:
    ax[1].plot(ep, [float(r[f"val_iou_{c}"]) for r in rows], label=c)
ax[1].plot(ep, [float(r["val_miou_fg"]) for r in rows], "k--", label="mIoU (leaf+disease)"); ax[1].set_title("validation IoU"); ax[1].legend()
for d in semseg.DISEASES:
    ax[2].plot(ep, [float(r[f"val_coverage_mae_{d}"]) for r in rows], label=d)
ax[2].set_title("validation coverage MAE (percentage points)"); ax[2].legend()
for a in ax:
    a.set_xlabel("epoch")

best_row = max(rows, key=lambda r: float(r["val_miou_fg"]))
print("best epoch:", best_row["epoch"], {k: round(float(v), 3) for k, v in best_row.items() if k.startswith("val_")})